# The CO2 tax in GREU with an FTT technology block

The whole GREU model (73 430 equations per year, 2021-2050, solved as one stacked system) with the FTT
technology block in place of DREAM's merit-order block. The shock is DREAM's S3: CO2 tax +750, phased in
2025-2030. Three runs are compared with the baseline:

* **DREAM block**: DREAM's own solution of S3. ModelFlow reproduces it from DREAM's staged start
  (max relative difference 6e-7).
* **FTT block**: the same shock on the model with the FTT block.
* **FTT block, tax until 2034**: the tax is removed after 2034.

With the calibrated terms the FTT model holds on DREAM's baseline, so all three runs are measured against the
same baseline. The technology block covers two units in the data: `in_ETS` in industry 23001 and `heating` in
industry 10030. The notebook `greu_ftt_block.ipynb` shows the block alone.

The runs are made by `python tools/ftt_block.py shock` and `python tools/ftt_block.py shock until=2034` (about 10
minutes each); this notebook only reads them.

*Example calculations based on example data. The model is GREU (ModelFlow translation) with an FTT block of our own, not DREAM's GreenREFORM-EU.*

In [1]:
import sys, re
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'tools'))
import greu_viz as gv
pd.set_option('display.width', 200)

In [2]:
def with_extra(df):
    extra = pd.DataFrame({'ENERGY_PJ': gv.energy_pj(df),                                    # physical energy use, PJ
                          'CCS_23001': -df['QEMME_CCS__IN_ETS__CAPTURED_CO2__23001']})      # captured CO2, positive
    return pd.concat([df, extra], axis=1)

base = with_extra(gv.solution('ftt_base_co2tax_et'))
RUNS = {'DREAM block': with_extra(gv.baseline('shock_carbon_tax_energy_technology')),
        'FTT block': with_extra(gv.solution('ftt_shock_co2tax_et')),
        'FTT block, tax until 2034': with_extra(gv.solution('ftt_shock_co2tax_et_until2034'))}
COLOR = {'DREAM block': gv.CAT[1], 'FTT block': gv.CAT[0], 'FTT block, tax until 2034': gv.CAT[2]}
DASH = {'DREAM block': 'dot'}
YEARS = list(range(2021, 2051))


def lines(vars_, title, kind='pct', cols=3, runs=None, with_base=False):
    """small multiples: one panel per variable, one line per run. kind: 'pct' = % difference from the baseline,
    'level' = the level (with the baseline in grey)"""
    runs = runs or RUNS
    rows = -(-len(vars_) // cols)
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=list(vars_.values()), horizontal_spacing=0.07,
                        vertical_spacing=0.16)
    for i, v in enumerate(vars_):
        r, c = i // cols + 1, i % cols + 1
        if kind == 'level':
            fig.add_trace(go.Scatter(x=YEARS, y=base.loc[YEARS, v], mode='lines', name='Baseline', legendgroup='Baseline',
                                     showlegend=(i == 0), line=dict(color=gv.GRAY, width=2),
                                     hovertemplate='Baseline %{x}: %{y:.4g}<extra></extra>'), row=r, col=c)
        for name, df in runs.items():
            y = gv.pct(df.loc[YEARS, v], base.loc[YEARS, v]) if kind == 'pct' else df.loc[YEARS, v]
            fig.add_trace(go.Scatter(x=YEARS, y=y, mode='lines', name=name, legendgroup=name, showlegend=(i == 0),
                                     line=dict(color=COLOR[name], width=2, dash=DASH.get(name)),
                                     hovertemplate=name + ' %{x}: %{y:.3f}<extra></extra>'), row=r, col=c)
    fig.update_layout(template=gv.TEMPLATE, height=120 + 260 * rows, hovermode='x unified', title=title,
                      legend=dict(orientation='h', y=-0.1 if rows > 1 else -0.22))
    fig.update_yaxes(zeroline=True, zerolinecolor='#d6d5d0')
    return fig


def table(vars_, years=(2025, 2030, 2040, 2050), kind='pct'):
    out = {}
    for v, lab in vars_.items():
        for name, df in RUNS.items():
            out[(lab, name)] = [gv.pct(df.at[y, v], base.at[y, v]) if kind == 'pct' else df.at[y, v] for y in years]
    return pd.DataFrame(out, index=list(years)).T

## The solve

Plain stacked Newton from the baseline, with MUMPS and a new Jacobian in every iteration. The chart shows the
largest residual by iteration, read from the logs. With DREAM's block the same plain solve from the baseline
diverges; that model is solved from DREAM's staged start instead (energy prices first, then the technology
shares, then the whole model).

The FTT model converges in 6 iterations for the permanent tax and 5 for the temporary one. The solves take
356 and 267 seconds, including finding the derivatives.

In [3]:
def residual_path(log):
    txt = (ROOT / 'log' / log).read_text(encoding='utf-8', errors='replace')
    return pd.Series({int(i): float(v) for i, v in re.findall(r'Iteration\s+(\d+) Max residual\s+([0-9.eE+-]+)', txt)})

paths = pd.DataFrame({'FTT block': residual_path('ftt_shock_co2tax_et.log'),
                      'FTT block, tax until 2034': residual_path('ftt_shock_co2tax_et_until2034.log')})
fig = go.Figure()
for name in paths:
    s = paths[name].dropna()
    fig.add_trace(go.Scatter(x=s.index, y=s.values, mode='lines+markers', name=name, line=dict(color=COLOR[name], width=2),
                             hovertemplate=name + ', iteration %{x}: %{y:.2e}<extra></extra>'))
fig.update_layout(template=gv.TEMPLATE, height=380, title='Largest residual by Newton iteration (log scale)',
                  xaxis_title='iteration', yaxis_type='log', yaxis_title='max residual', xaxis=dict(dtick=1))
fig.show()
paths.T

,0,1,2,3,4,5,6
FTT block,4.744532,14.427495,0.639781,0.257388,0.026017,7.264154e-05,1.828960e-07
"FTT block, tax until 2034",4.744532,5.464086,0.145409,0.003477,0.000012,3.740300e-08,NaN


## The technology shares in the whole model

The shares in `in_ETS`, industry 23001, where the tax makes coal with CCS (t11) cheaper.

* **DREAM block**: coal with CCS is at its potential of 12.5 % from 2028. Gasoline falls from 12.2 % to 7.0 %.
* **FTT block**: coal with CCS reaches 4.4 % in 2030 and 7.2 % in 2050. Every other technology gives up a little.
* **Tax until 2034**: coal with CCS peaks at 5.3 % in 2034 and is still at 4.2 % in 2050 (baseline 3.1 %).

The shares are the same to two decimals as when the block runs alone with DREAM's prices: the feedback from the
rest of the model to the shares is small.

In [4]:
UNIT = 'IN_ETS__23001'
lab = gv.technology_labels(base, UNIT)
lab = {l: f'{l}: coal with CCS' if 'captured co2' in t else t for l, t in lab.items()}
sh = {f'SQT__{t.upper()}__{UNIT}': lab[t] for t in ('t11', 't13', 't12', 't8', 't2', 't7')}
fig = lines(sh, 'Technology shares, in_ETS, industry 23001 (share of the energy service)', kind='level')
fig.update_yaxes(rangemode='tozero')
fig.show()
(100 * table(sh, kind='level')).round(2)

2025   2030   2040   2050
t11: coal with CCS         DREAM block                 5.28  12.50  12.50  12.50
                           FTT block                   3.27   4.42   6.32   7.19
                           FTT block, tax until 2034   3.27   4.42   4.73   4.15
t13: coal and coke         DREAM block                11.41  10.00  10.00  10.01
                           FTT block                  12.28  12.13  11.85  11.73
                           FTT block, tax until 2034  12.28  12.13  12.07  12.15
t12: coal and coke         DREAM block                11.96  11.28  11.28  11.28
                           FTT block                  12.37  12.22  11.98  11.88
                           FTT block, tax until 2034  12.37  12.22  12.18  12.26
t8: coal and coke          DREAM block                12.18  11.86  11.86  11.86
                           FTT block                  12.39  12.24  12.01  11.91
                           FTT block, tax until 2034  12.39  12.24  12.21  12.29
t2: gasoline for transport DREAM block                11.71   6.99   6.99   6.99
                           FTT block                  12.17  12.01  11.69  11.48
                           FTT block, tax until 2034  12.17  12.01  11.91  11.98
t7: waste                  DREAM block                12.50  12.50  12.50  12.50
                           FTT block                  12.48  12.34  12.15  12.09
                           FTT block, tax until 2034  12.48  12.34  12.34  12.41

## The industry that uses the technologies

Industry 23001: output, the price of the energy service, capital in the technologies, coal use and captured CO2.

* Output falls by about 11 % with both blocks (-10.9 % in DREAM's and -11.4 % in FTT in 2030). The fall comes from
  the rest of the model, not from the choice of technology.
* The price of the energy service falls less with FTT (-2.3 % against -5.9 % in 2030), as the cheaper technology
  comes in slowly.
* Captured CO2 is up 260 % from 2030 in DREAM's block. With FTT it is up 25 % in 2030 and 104 % in 2050.
* With DREAM's block capital in the technologies rises 7.5 % (CCS is capital heavy) and coal use rises. With FTT
  both fall with the energy service, as the shift to CCS is small in the first years.

In [5]:
IND = {'QY_I__23001': 'Output, industry 23001', 'PES__IN_ETS__23001': 'Price of the energy service in_ETS',
       'QES__IN_ETS__23001': 'Energy service in_ETS, quantity', 'QESK__IN_ETS__23001': 'Capital in the technologies',
       'QESE__IN_ETS__COAL_AND_COKE__23001': 'Coal and coke in in_ETS, PJ', 'CCS_23001': 'Captured CO2 (CCS)'}
lines(IND, 'Industry 23001, % difference from the baseline').show()
table(IND).round(2)

2025    2030    2040    2050
Output, industry 23001             DREAM block                -1.85  -10.90  -11.12  -11.27
                                   FTT block                  -1.89  -11.35  -11.48  -11.59
                                   FTT block, tax until 2034  -1.91  -11.35   -0.04   -0.04
Price of the energy service in_ETS DREAM block                -0.39   -5.92   -5.92   -5.91
                                   FTT block                  -0.29   -2.31   -3.01   -3.34
                                   FTT block, tax until 2034  -0.29   -2.31    0.36    0.21
Energy service in_ETS, quantity    DREAM block                -1.78   -9.52   -9.30   -9.44
                                   FTT block                  -1.81  -11.09  -10.58  -10.58
                                   FTT block, tax until 2034  -1.84  -11.13   -0.03   -0.09
Capital in the technologies        DREAM block                -3.00    7.52    7.79    7.57
                                   FTT block                  -1.55   -8.78   -4.82   -3.24
                                   FTT block, tax until 2034  -1.58   -8.82    3.19    1.95
Coal and coke in in_ETS, PJ        DREAM block                -0.33    2.65    2.89    2.74
                                   FTT block                  -1.61   -9.35   -6.28   -5.11
                                   FTT block, tax until 2034  -1.64   -9.39    2.36    1.42
Captured CO2 (CCS)                 DREAM block                64.87  260.02  260.89  260.03
                                   FTT block                   2.27   25.10   79.99  104.47
                                   FTT block, tax until 2034   2.24   25.05   50.57   31.94

## Energy and emissions in the whole economy

The technology block covers two industries, so the totals are close: emissions are down 5.60 % in 2030 with FTT
and 5.67 % with DREAM's block (2 351 and 2 382 kilotonnes). When the tax ends, emissions are back within 0.1 % of
the baseline the year after.

In [6]:
EN = {'QEMMTOT__CO2E__UNFCCC': 'Greenhouse gas emissions (CO2e)', 'ENERGY_PJ': 'Energy use, PJ',
      'QE': 'Real energy inputs (CGE volume)'}
lines(EN, 'Energy and emissions, % difference from the baseline').show()
t = table(EN).round(3)
dd = pd.DataFrame({name: (df.loc[[2025, 2030, 2040, 2050], 'QEMMTOT__CO2E__UNFCCC'] - base.loc[[2025, 2030, 2040, 2050], 'QEMMTOT__CO2E__UNFCCC'])
                   for name, df in RUNS.items()}).T
dd.index = pd.MultiIndex.from_product([['CO2e, change in kilotonnes'], dd.index])
pd.concat([t, dd.round(0)])

2025      2030      2040      2050
Greenhouse gas emissions (CO2e) DREAM block                 -1.087    -5.674    -5.778    -5.906
                                FTT block                   -1.054    -5.600    -5.720    -5.855
                                FTT block, tax until 2034   -1.066    -5.609    -0.061    -0.023
Energy use, PJ                  DREAM block                 -0.339    -1.747    -1.813    -1.851
                                FTT block                   -0.340    -1.759    -1.823    -1.860
                                FTT block, tax until 2034   -0.338    -1.757    -0.029    -0.002
Real energy inputs (CGE volume) DREAM block                 -0.740    -3.749    -3.831    -3.874
                                FTT block                   -0.741    -3.764    -3.845    -3.887
                                FTT block, tax until 2034   -0.729    -3.754    -0.033    -0.000
CO2e, change in kilotonnes      DREAM block               -457.000 -2382.000 -2431.000 -2502.000
                                FTT block                 -443.000 -2351.000 -2407.000 -2480.000
                                FTT block, tax until 2034 -448.000 -2355.000   -26.000   -10.000

## The macro economy

The macro effects are the same with the two blocks to two decimals. The temporary tax shows the dynamics of the
rest of the model: the economy is forward-looking, so consumption and exports differ from the permanent tax
already in 2025, and investment overshoots when the tax ends (+0.6 % in 2035) while the capital stock is rebuilt.

In [7]:
MACRO = {'QGDP': 'Real GDP', 'QC': 'Private consumption', 'QI': 'Investment', 'QX': 'Exports', 'QM': 'Imports',
         'PW': 'Wage'}
lines(MACRO, 'Macro economy, % difference from the baseline').show()
table(MACRO).round(3)

2025   2030   2040   2050
Real GDP            DREAM block               -0.012 -0.263 -0.349 -0.362
                    FTT block                 -0.012 -0.264 -0.350 -0.363
                    FTT block, tax until 2034 -0.025 -0.281 -0.031 -0.003
Private consumption DREAM block                0.096 -0.140 -0.295 -0.400
                    FTT block                  0.096 -0.139 -0.294 -0.399
                    FTT block, tax until 2034  0.005 -0.228 -0.020  0.002
Investment          DREAM block               -0.021 -0.767 -0.392 -0.362
                    FTT block                 -0.021 -0.773 -0.394 -0.364
                    FTT block, tax until 2034 -0.088 -0.825  0.191  0.007
Exports             DREAM block               -0.190 -0.528 -0.750 -0.711
                    FTT block                 -0.190 -0.529 -0.753 -0.714
                    FTT block, tax until 2034 -0.107 -0.464 -0.133  0.011
Imports             DREAM block               -0.095 -0.538 -0.579 -0.545
                    FTT block                 -0.095 -0.539 -0.580 -0.546
                    FTT block, tax until 2034 -0.099 -0.543 -0.017  0.011
Wage                DREAM block                0.029 -0.249 -0.256 -0.266
                    FTT block                  0.029 -0.251 -0.257 -0.266
                    FTT block, tax until 2034 -0.016 -0.288  0.001 -0.009

## After the tax ends

The run with the tax removed after 2034: how much of the effect is left. Shares are in % of the energy service;
the other rows are % difference from the baseline.

The technology share is the slow variable. Emissions, output and GDP are back near the baseline within a few
years, but coal with CCS keeps a higher share, so captured CO2 is still 32 % above the baseline in 2050. Captured
CO2 rises in 2035, because the industry's output recovers while the share is still high.

In [8]:
tmp = RUNS['FTT block, tax until 2034']
ys = [2034, 2035, 2036, 2040, 2045, 2050]
rows = {'Share of coal with CCS (t11), in_ETS, %': 100 * tmp.loc[ys, 'SQT__T11__IN_ETS__23001'],
        '   baseline, %': 100 * base.loc[ys, 'SQT__T11__IN_ETS__23001']}
for v, l in {'CCS_23001': 'Captured CO2', 'QEMMTOT__CO2E__UNFCCC': 'Greenhouse gas emissions', 'QY_I__23001': 'Output, industry 23001',
             'QGDP': 'Real GDP', 'QI': 'Investment'}.items():
    rows[l + ', % from baseline'] = gv.pct(tmp.loc[ys, v], base.loc[ys, v])
pd.DataFrame(rows).T.round(3)

year,2034,2035,2036,2040,2045,2050
"Share of coal with CCS (t11), in_ETS, %",5.305,5.191,5.086,4.731,4.401,4.152
"baseline, %",3.141,3.141,3.141,3.141,3.143,3.144
"Captured CO2, % from baseline",50.063,66.378,62.427,50.568,39.863,31.938
"Greenhouse gas emissions, % from baseline",-5.655,-0.098,-0.111,-0.061,-0.034,-0.023
"Output, industry 23001, % from baseline",-11.425,-0.371,-0.268,-0.045,-0.042,-0.038
"Real GDP, % from baseline",-0.361,-0.076,-0.097,-0.031,-0.009,-0.003
"Investment, % from baseline",-0.765,0.615,0.378,0.191,0.042,0.007
